# Examen bimestral

**Introducción a los Sistemas de Información**

Prof. Iván Carrera

Ingeniería en Ciencia de Datos e Inteligencia Artificial · Escuela Politécnica Nacional

**Calificación total:** 10 puntos.

## Caso de estudio · Papelería universitaria

Una papelería universitaria registra las compras de sus sucursales Central, Norte y Sur en un sistema de punto de venta. Cada compra genera un ticket que puede incluir varios productos. La persona encargada de la papelería necesita resumir estos registros para conocer qué se vendió, cómo se pagaron las compras y cómo variaron las ventas entre sucursales y meses.

Elabore los reportes solicitados a partir de los datos.

## Archivos de trabajo

| Archivo      | Contenido |
|--------------|---|
| `productos.csv` | Una fila por producto. Incluye su código, nombre, categoría y precio de lista. |
| `clientes.csv` | Una fila por cliente. Incluye su identificador, nombre, ciudad y tipo de cliente. |
| `ventas.csv`   | Una fila por producto dentro de un ticket. Incluye la fecha, sucursal, cliente, cantidad, precio unitario y método de pago. |

El campo `id_ticket` puede repetirse en varias filas de `ventas.csv`. Para relacionar los archivos, use `id_producto` en ventas y productos, y `ventas.id_cliente` con `clientes.id`. Un campo vacío en `metodo_pago` indica que ese dato quedó sin registrar.

## Preguntas

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

## Entrega y evaluación

Resuelva los reportes solicitados en un mismo Jupyter Notebook, identificados con el número de cada pregunta. Presente los reportes con encabezados claros y los importes en dólares con dos decimales. Use conteos, agrupaciones y cálculos descriptivos.

Entregue el link al Jupyter Notebook cargado en un repositorio Git en la Tarea del aula virtual.

**Cada pregunta vale 1 punto, para un total de 10 puntos.** En cada respuesta se evaluará que el código produzca el resultado solicitado a partir de los CSV y que se presenten todos los elementos pedidos en la pregunta.

Buena suerte.

In [37]:
import pandas as pd

df_C= pd.read_csv('clientes.csv')
df_P= pd.read_csv('productos_.csv')
df_V= pd.read_csv('ventas.csv')


print(df_C.head(2))
print(df_P.head(2))
print(df_V.head(2))



     id          nombre ciudad tipo_cliente
0  C001  Sofía Villacís  Quito      Docente
1  C002     Daniel Vega  Quito      Docente
  id_producto              producto  categoria  precio_lista
0         P07            Resaltador  Escritura           1.1
1         P12  Paquete de fundas A4    Archivo           2.1
  id_linea id_ticket       fecha sucursal id_cliente id_producto  cantidad  \
0   L00001     V0001  2025-11-01      Sur       C089        P087         1   
1   L00002     V0001  2025-11-01      Sur       C089        P326         4   

   precio_unitario    metodo_pago  
0             1.50  Transferencia  
1             0.93  Transferencia  


## Nota: Cada modificación realizada en el código está comentada

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [47]:
#Filas del archivo: representa cada producto dentro de un ticket
ventasLin = len(df_V) 

#Compras (tickets) únicas; un ticket puede tener varias líneas
ticketDistinto = df_V["id_ticket"].nunique()

#Suma de la columna cantidad: piezas físicas vendidas
unidadesVen=df_V["cantidad"].sum()

#unidadesVendidas=sum(df_V["cantidad"])
# CAMBIO: se eliminó unidadesVendidas = sum(...) porque calculaba lo mismo que unidadesVen (código repetido).


tablaVentas = pd.DataFrame({
    "Categoría": ["Líneas de venta", "Tickets distintos", "Unidades vendidas"],
    "Valor": [ventasLin, ticketDistinto, unidadesVen]
})
display(tablaVentas)

,Categoría,Valor
0,Líneas de venta,2100
1,Tickets distintos,300
2,Unidades vendidas,3880


2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [63]:
ticketSur=0
ticketNorte=0
ticketCentral=0
#Guarda los tickets que ya se contaron
ticketsVistos = [] 

for sucursal, ticket in zip(df_V["sucursal"], df_V["id_ticket"]):   #Añado la fila de tickets para analizarlo
     if ticket not in ticketsVistos:      #Solo cuenta el ticket la primera vez que aparece
        ticketsVistos.append(ticket)
        if sucursal == "Norte":
            ticketNorte += 1
        elif sucursal == "Sur":
            ticketSur += 1
        else:
            ticketCentral += 1



masRegistrado="" #Modifiqué el nombre porque usaba otro en el resto del código
if ticketSur>ticketNorte:
    if ticketSur > ticketCentral:
        masRegistrado="Sucursal Sur"
    else:
        masRegistrado="Sucursal Central"
elif ticketNorte > ticketCentral:
    masRegistrado="Sucursal Norte"
else:
    masRegistrado="Sucursal Central"

tablaSucursal = pd.DataFrame({
    "Sucursales": ["Central", "Norte", "Sur", "Con más registros"],
    "Tickets": [ticketCentral,ticketNorte, ticketSur, masRegistrado] #Cambié el primer dato porque estaba con otro nombre
})
display(tablaSucursal)


,Sucursales,Tickets
0,Central,146
1,Norte,83
2,Sur,71
3,Con más registros,Sucursal Central



3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [71]:
ticketTrans=0
ticketEfec=0
ticketTar=0
ticketSR=0
#Para los tickets que ya se contaron
ticketsVistos = []   

for ticket, metodo in zip(df_V["id_ticket"], df_V["metodo_pago"]): #Añado la fila de tickets para analizar
    if ticket not in ticketsVistos:      # solo cuenta el ticket la primera vez que aparece
        ticketsVistos.append(ticket)     
        if metodo == "Transferencia":
            ticketTrans += 1
        elif metodo == "Efectivo":
            ticketEfec += 1
        elif metodo == "Tarjeta":
            ticketTar += 1
        else:
            ticketSR += 1             
    

tablaMetodo=pd.DataFrame({
    "Metodo": ["Efectivo", "Transferencia", "Tarjeta", "Sin Registrar"],
    "Cantidad de Tickets": [ticketEfec, ticketTrans, ticketTar, ticketSR ]
})

display(tablaMetodo)


,Metodo,Cantidad de Tickets
0,Efectivo,144
1,Transferencia,48
2,Tarjeta,84
3,Sin Registrar,24



4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [106]:
#Está correcto el código
escritura=[]
papelYCuadernos=[]
archivo=[]
accesorios=[]

for categoria, precio in zip(df_P["categoria"], df_P["precio_lista"]):
    if categoria == "Escritura":
        escritura.append(precio)

    elif categoria == "Papel y cuadernos":
        papelYCuadernos.append(precio)
    elif categoria == "Archivo":
        archivo.append(precio)
    else:
        accesorios.append(precio)
        


promedio1 = sum(escritura)/len(escritura)
maxim1 = max(escritura)
minim1 = min(escritura)

promedio2 = sum(papelYCuadernos)/len(papelYCuadernos)
maxim2 = max(papelYCuadernos)
minim2 = min(papelYCuadernos)

promedio3 = sum(archivo)/len(archivo)
maxim3 = max(archivo)
minim3 = min(archivo)

promedio4 = sum(accesorios)/len(accesorios)
maxim4 = max(accesorios)
minim4 = min(accesorios)

tablaListaCat = pd.DataFrame({
    "Categoria": ["Escritura", "Papel y Cuadernos", "Archivo", "Accesorios" ],
    "Mínimo": [f"${minim1:.2f}", f"${minim2:.2f}", f"${minim3:.2f}", f"${minim4:.2f}"],
    "Máximo": [f"${maxim1:.2f}", f"${maxim2:.2f}", f"${maxim3:.2f}", f"${maxim4:.2f}"],
    "Promedio": [f"${promedio1:.2f}", f"${promedio2:.2f}", f"${promedio3:.2f}", f"${promedio4:.2f}"]

}) #Modifiqué la tabla para que salga con dos decimales y el signo de dólar

display(tablaListaCat)

,Categoria,Mínimo,Máximo,Promedio
0,Escritura,$0.40,$3.75,$1.35
1,Papel y Cuadernos,$0.28,$5.75,$1.67
2,Archivo,$0.35,$3.95,$1.67
3,Accesorios,$0.40,$22.90,$1.62


5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

In [107]:
#Reestructuración de todo el código para poder crear la tabla cruzada solicitada

# Diccionario id_producto -> categoría (la categoría solo está en productos.csv)
categoriaDe = {}
for idproducto, categoria in zip(df_P["id_producto"], df_P["categoria"]):
    categoriaDe[idproducto] = categoria

sucursales = ["Norte", "Central", "Sur"]
categorias = ["Escritura", "Papel y cuadernos", "Archivo", "Accesorios"]

# Celda (categoría, sucursal) iniciada en 0
unidades = {}
for categoria in categorias:
    for sucursal in sucursales:
        unidades[(categoria, sucursal)] = 0

# Se suman las unidades (cantidad), no las líneas
for sucursal, idproducto, cantidad in zip(df_V["sucursal"], df_V["id_producto"], df_V["cantidad"]):
    categoria = categoriaDe[idproducto]
    unidades[(categoria, sucursal)] += cantidad

# Columnas de la tabla y total por fila
colNorte, colCentral, colSur, colTotal = [], [], [], []
for categoria in categorias:
    norte = unidades[(categoria, "Norte")]
    central = unidades[(categoria, "Central")]
    sur = unidades[(categoria, "Sur")]
    colNorte.append(norte)
    colCentral.append(central)
    colSur.append(sur)
    colTotal.append(norte + central + sur)

# Fila de totales por columna y total general
categorias_tabla = categorias + ["Total"]
colNorte.append(sum(colNorte))
colCentral.append(sum(colCentral))
colSur.append(sum(colSur))
colTotal.append(sum(colTotal))

tablaVSucursalCateg = pd.DataFrame({
    "Categoría": categorias_tabla,
    "Norte": colNorte,
    "Central": colCentral,
    "Sur": colSur,
    "Total": colTotal
})
display(tablaVSucursalCateg)


,Categoría,Norte,Central,Sur,Total
0,Escritura,306,437,238,981
1,Papel y cuadernos,259,524,248,1031
2,Archivo,264,481,223,968
3,Accesorios,256,453,191,900
4,Total,1085,1895,900,3880


6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

In [113]:
#Está correcto el código

ventasSur=[]
ventasNorte=[]
ventasCentral=[]

for sucursal, precio_unitario, cantidad in zip(df_V["sucursal"], df_V["precio_unitario"], df_V["cantidad"]):
    if sucursal == "Norte":
        ventasNorte.append((precio_unitario * cantidad))
    elif sucursal == "Sur":
        ventasSur.append((precio_unitario * cantidad))
    else:
        ventasCentral.append((precio_unitario * cantidad))

ventasTSur=sum(ventasSur)
ventasTNorte=sum(ventasNorte)
ventasTCentral=sum(ventasCentral)

tablaVSuc= pd.DataFrame({
    "Sucursal": ["Norte", "Sur", "Central"],
    "Total Ventas ($)": [f"${ventasTNorte:.2f}", f"${ventasTSur:.2f}", f"${ventasTCentral:.2f}"] 
    #Cambié el orden de las variables porque estaba alrevés
    #Modifiqué la tabla para que salga con dos decimales y el signo de dólar
})

display(tablaVSuc)

,Sucursal,Total Ventas ($)
0,Norte,$1635.35
1,Sur,$1443.24
2,Central,$3025.04


7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

In [114]:
#Está correcto el código

ticketCant={}
hasta7=0
de8_14=0
de15oMas=0

for ticket, cantidad in zip (df_V["id_ticket"], df_V["cantidad"]):
    if ticket not in ticketCant:
        ticketCant[ticket]=0

    ticketCant[ticket] += cantidad


for ticket, total in ticketCant.items():
    if total<=7:
        hasta7+=1
    elif total <= 14:
        de8_14 +=1
    else:
        de15oMas+=1


tablaTickets=pd.DataFrame({
    "Unidades": ["Hasta 7", "De 8 a 14", "15 o más"],
    "Num Tickets" :[hasta7, de8_14, de15oMas ]
})

display(tablaTickets)

,Unidades,Num Tickets
0,Hasta 7,39
1,De 8 a 14,153
2,15 o más,108



8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [ ]:
#Está correcto el código

conteoTickets=df_V["id_ticket"].value_counts()
cantidadTickets= len(conteoTickets)


mesestickets={}
mesesimportes={}


tickMes1=0
for fecha, ticket, cant, precio  in zip(df_V["fecha"], df_V["id_ticket"], df_V["cantidad"], df_V["precio_unitario"]):
    mes= str(fecha)[:7]
    importe = cant * precio

    if mes not in mesesimportes:
        mesesimportes[mes]=0
        mesestickets[mes]= set()

    mesesimportes[mes]+= importe
    mesestickets[mes].add(ticket)


lista_meses = sorted(mesesimportes.keys())
lista_tickets = [
    len(mesestickets[m]) for m in lista_meses
]  
lista_importes = [round(mesesimportes[m], 2) for m in lista_meses]

tablaVentasMes = pd.DataFrame({
    "Mes": lista_meses,
    "Tickets": lista_tickets,
    "Importe Total ($)": [f"${importe:.2f}" for importe in lista_importes],
    #Modifiqué la fila para que salga con dos decimales y el signo de dólar
})
display(tablaVentasMes)

mayor_importe = max(lista_importes)
indice_mayor = lista_importes.index(mayor_importe)

print("Mes con mayor importe:", lista_meses[indice_mayor], "con $" + format(mayor_importe, ".2f")) 
#Mandé a imprimir el mes con mayor importe, símbolo de dólar y redondeado a dos decimales
 

,Mes,Tickets,Importe Total ($)
0,2025-11,25,$492.51
1,2025-12,25,$645.54
2,2026-01,25,$447.48
3,2026-02,25,$466.36
4,2026-03,25,$595.33
5,2026-04,25,$505.43
6,2026-05,25,$453.94
7,2026-06,25,$500.54
8,2026-07,25,$499.88
9,2026-08,25,$478.79


Mes con mayor importe: 2025-12 con $645.54



9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [122]:
#Realicé una reestructuración del código 

# df_V(df_P, on=..., how='left') daba TypeError porque df_V es un DataFrame, no una función;
# faltaba el .merge. Para mantener el código simple (sin merge) se usa un diccionario id_producto -> nombre,
# igual que en la pregunta 5.
nombreDe = {}
for idproducto, nombre in zip(df_P["id_producto"], df_P["producto"]):
    nombreDe[idproducto] = nombre

sucursales = ["Central", "Norte", "Sur"]
listaSucursal = []
listaIdProd = []
listaNombreProd = []
listaCantidades = []

for sucursal in sucursales:
    dic_cantidades = {}

    # CAMBIO: se recorre df_V directamente filtrando por sucursal con un if (no hace falta el df_suc intermedio).
    for suc, idProducto, cant in zip(df_V["sucursal"], df_V["id_producto"], df_V["cantidad"]):
        if suc == sucursal:
            dic_cantidades[idProducto] = dic_cantidades.get(idProducto, 0) + cant

    max_cant = max(dic_cantidades.values())

    # La lógica de empates del original estaba bien y se mantiene:
    # se agregan todos los productos que igualan el máximo.
    for idProducto, cantidad in dic_cantidades.items():
        if cantidad == max_cant:
            listaSucursal.append(sucursal)
            listaIdProd.append(idProducto)
            listaNombreProd.append(nombreDe[idProducto])
            listaCantidades.append(cantidad)

tabla_top = pd.DataFrame({
    "Sucursal": listaSucursal,
    "Código": listaIdProd,
    "Producto": listaNombreProd,
    "Cantidad Vendida": listaCantidades
})
display(tabla_top)


,Sucursal,Código,Producto,Cantidad Vendida
0,Central,P399,Estuche escolar (gris),17
1,Norte,P141,Marcador de punta fina (naranja),17
2,Sur,P304,Portadocumentos con cierre (rojo),11
3,Sur,P016,Cuaderno cuadriculado A5 (24 hojas),11



10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [123]:
# Reestructuración commpleta del código 


# Diccionario id_cliente -> tipo de cliente
tipoDeCliente = {}
for idCliente, tipoCliente in zip(df_C["id"], df_C["tipo_cliente"]):
    tipoDeCliente[idCliente] = tipoCliente

tiposCliente = ["Docente", "Estudiante", "Personal administrativo"]
listaTipoCliente = []
listaClientesDistintos = []
listaTicketsGenerados = []
listaImporteTotal = []

for tipoActual in tiposCliente:
    clientesContados = []     # clientes ya contados de este tipo
    ticketsContados = []      # tickets ya contados de este tipo
    sumaImporte = 0

    for idCliente, idTicket, cantidad, precioUnitario in zip(df_V["id_cliente"], df_V["id_ticket"], df_V["cantidad"], df_V["precio_unitario"]):
        if tipoDeCliente[idCliente] == tipoActual:
            sumaImporte += cantidad * precioUnitario     # importe de la línea
            if idCliente not in clientesContados:
                clientesContados.append(idCliente)
            if idTicket not in ticketsContados:
                ticketsContados.append(idTicket)

    listaTipoCliente.append(tipoActual)
    listaClientesDistintos.append(len(clientesContados))
    listaTicketsGenerados.append(len(ticketsContados))
    listaImporteTotal.append(sumaImporte)

tablaClientes = pd.DataFrame({
    "Tipo de Cliente": listaTipoCliente,
    "Clientes Distintos": listaClientesDistintos,
    "Tickets Generados": listaTicketsGenerados,
    "Importe Total ($)": [f"${importe:.2f}" for importe in listaImporteTotal] #Para tener el formato de 2 decimales
})

display(tablaClientes)

,Tipo de Cliente,Clientes Distintos,Tickets Generados,Importe Total ($)
0,Docente,16,46,$899.74
1,Estudiante,72,226,$4688.00
2,Personal administrativo,12,28,$515.89
